# Linear Algebra from Scratch

**Module 00 — Mathematical Foundations for AI**

This notebook implements core linear algebra operations using only NumPy.
Every operation here is what happens inside PyTorch/TensorFlow under the hood.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Vectors and Basic Operations

Vectors are the atomic unit of data in ML. Every data point, every weight, every gradient is a vector.

In [ ]:
# Creating vectors
v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([4.0, 5.0, 6.0])

# Dot product: fundamental operation in neural networks
# Every neuron computes: output = dot(weights, inputs) + bias
def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    """Compute dot product from scratch."""
    assert len(a) == len(b), "Vectors must have same length"
    result = 0.0
    for i in range(len(a)):
        result += a[i] * b[i]
    return result

print(f"Manual dot product: {dot_product(v1, v2)}")
print(f"NumPy dot product:  {np.dot(v1, v2)}")
print(f"Equivalent: v1 @ v2 = {v1 @ v2}")

In [ ]:
# Norms: measuring vector magnitude
def l1_norm(v: np.ndarray) -> float:
    """Manhattan distance - used in L1 regularization (Lasso)."""
    return np.sum(np.abs(v))

def l2_norm(v: np.ndarray) -> float:
    """Euclidean distance - used in L2 regularization (Ridge), cosine similarity."""
    return np.sqrt(np.sum(v ** 2))

def linf_norm(v: np.ndarray) -> float:
    """Max norm - used in adversarial robustness."""
    return np.max(np.abs(v))

v = np.array([3.0, -4.0, 0.0])
print(f"L1 norm:  {l1_norm(v):.4f} (NumPy: {np.linalg.norm(v, 1):.4f})")
print(f"L2 norm:  {l2_norm(v):.4f} (NumPy: {np.linalg.norm(v, 2):.4f})")
print(f"L∞ norm: {linf_norm(v):.4f} (NumPy: {np.linalg.norm(v, np.inf):.4f})")

In [ ]:
# Cosine similarity: THE metric for embedding comparison
# Used in: semantic search, recommendation systems, RAG retrieval
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Example: word embeddings (simplified)
king = np.array([0.5, 0.8, 0.1, 0.3])
queen = np.array([0.5, 0.7, 0.9, 0.3])
car = np.array([0.1, 0.1, 0.2, 0.9])

print(f"Similarity(king, queen): {cosine_similarity(king, queen):.4f}")
print(f"Similarity(king, car):   {cosine_similarity(king, car):.4f}")
print("→ king and queen are more similar than king and car")

## 2. Matrix Operations

Every layer in a neural network is a matrix multiplication.
Understanding matrix ops = understanding what neural networks compute.

In [ ]:
def matmul(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Matrix multiplication from scratch.
    
    This is O(m*n*p) - the reason we need GPUs.
    A neural network with L layers does L matrix multiplications per forward pass.
    For a 175B parameter model (GPT-3), this is ~350 TFLOPS per token.
    """
    assert A.shape[1] == B.shape[0], f"Incompatible shapes: {A.shape} x {B.shape}"
    m, n = A.shape
    _, p = B.shape
    C = np.zeros((m, p))
    for i in range(m):
        for j in range(p):
            for k in range(n):
                C[i, j] += A[i, k] * B[k, j]
    return C

A = np.array([[1, 2], [3, 4], [5, 6]])  # 3x2
B = np.array([[7, 8, 9], [10, 11, 12]])  # 2x3

manual_result = matmul(A, B)
numpy_result = A @ B

print("Manual matmul:")
print(manual_result)
print("\nNumPy matmul:")
print(numpy_result)
print(f"\nMatch: {np.allclose(manual_result, numpy_result)}")

In [ ]:
# Simulating a neural network layer
# y = Wx + b (single layer, no activation)

batch_size = 4
input_dim = 3
output_dim = 2

# Random input batch (4 samples, 3 features each)
X = np.random.randn(batch_size, input_dim)

# Weight matrix and bias (what the network learns)
W = np.random.randn(input_dim, output_dim)  # 3x2
b = np.random.randn(1, output_dim)  # 1x2 (broadcasted)

# Forward pass: this is ALL a linear layer does
Y = X @ W + b  # (4x3) @ (3x2) + (1x2) = (4x2)

print(f"Input shape:  {X.shape}  (batch_size x input_dim)")
print(f"Weight shape: {W.shape}  (input_dim x output_dim)")
print(f"Output shape: {Y.shape}  (batch_size x output_dim)")
print(f"\nOutput:\n{Y}")

## 3. Eigenvalues and Eigenvectors

Eigendecomposition reveals the fundamental structure of a matrix.
Used in: PCA, spectral clustering, understanding neural network training dynamics.

In [ ]:
def power_iteration(A: np.ndarray, num_iters: int = 100) -> Tuple[float, np.ndarray]:
    """Find the dominant eigenvalue and eigenvector using power iteration.
    
    This is how Google's original PageRank worked!
    Complexity: O(n^2) per iteration (matrix-vector multiply)
    """
    n = A.shape[0]
    v = np.random.randn(n)
    v = v / np.linalg.norm(v)
    
    for _ in range(num_iters):
        # Matrix-vector multiply
        Av = A @ v
        # Normalize
        v_new = Av / np.linalg.norm(Av)
        v = v_new
    
    # Eigenvalue via Rayleigh quotient
    eigenvalue = (v @ A @ v) / (v @ v)
    return eigenvalue, v

# Test with a symmetric matrix
A = np.array([[4.0, 1.0], [1.0, 3.0]])

# Our implementation
val, vec = power_iteration(A)
print(f"Power iteration: eigenvalue = {val:.4f}, eigenvector = {vec}")

# NumPy verification
eigenvalues, eigenvectors = np.linalg.eig(A)
print(f"NumPy:           eigenvalues = {eigenvalues}")
print(f"                 eigenvectors = \n{eigenvectors}")

In [ ]:
# Visualize eigenvectors: they show the principal axes of the data
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Generate correlated 2D data
mean = [0, 0]
cov = [[3, 2], [2, 2]]  # covariance matrix
data = np.random.multivariate_normal(mean, cov, 200)

# Compute eigenvectors of covariance
cov_matrix = np.cov(data.T)
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Sort by eigenvalue (largest first)
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Plot data with eigenvectors
axes[0].scatter(data[:, 0], data[:, 1], alpha=0.5, s=20)
origin = np.mean(data, axis=0)
for i, (val, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    axes[0].annotate('', xy=origin + vec * val * 2, xytext=origin,
                     arrowprops=dict(arrowstyle='->', color=['red', 'blue'][i], lw=3))
axes[0].set_title('Data with Principal Components (Eigenvectors)')
axes[0].set_xlabel('x₁')
axes[0].set_ylabel('x₂')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Transform data to principal component space
data_centered = data - np.mean(data, axis=0)
data_transformed = data_centered @ eigenvectors

axes[1].scatter(data_transformed[:, 0], data_transformed[:, 1], alpha=0.5, s=20)
axes[1].set_title('Data in Principal Component Space')
axes[1].set_xlabel('PC1 (direction of max variance)')
axes[1].set_ylabel('PC2')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../diagrams/pca_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Variance explained: PC1={eigenvalues[0]:.2f}, PC2={eigenvalues[1]:.2f}")
print(f"PC1 explains {eigenvalues[0]/sum(eigenvalues)*100:.1f}% of variance")

## 4. Singular Value Decomposition (SVD)

SVD is the Swiss Army knife of linear algebra.
Used in: PCA, LoRA (the most important fine-tuning technique), matrix completion, LSA.

In [ ]:
def svd_from_scratch(A: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute SVD via eigendecomposition of A^T A and A A^T.
    
    A = U Σ V^T
    
    The key insight: A^T A = V Σ^2 V^T, so V comes from eigendecomposition of A^T A.
    Similarly, A A^T = U Σ^2 U^T.
    """
    # Compute A^T A
    ATA = A.T @ A
    
    # Eigendecomposition of A^T A gives V and σ²
    eigenvalues, V = np.linalg.eigh(ATA)
    
    # Sort in descending order
    idx = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[idx]
    V = V[:, idx]
    
    # Singular values are square roots of eigenvalues
    singular_values = np.sqrt(np.maximum(eigenvalues, 0))
    
    # Compute U = A V Σ^{-1}
    rank = np.sum(singular_values > 1e-10)
    U = np.zeros((A.shape[0], rank))
    for i in range(rank):
        U[:, i] = (A @ V[:, i]) / singular_values[i]
    
    return U, singular_values[:rank], V[:, :rank].T

# Test
A = np.array([[1, 2], [3, 4], [5, 6]], dtype=float)

U_manual, s_manual, Vt_manual = svd_from_scratch(A)
U_numpy, s_numpy, Vt_numpy = np.linalg.svd(A, full_matrices=False)

print("Singular values (manual):", s_manual)
print("Singular values (NumPy): ", s_numpy)

# Verify reconstruction
A_reconstructed = U_manual @ np.diag(s_manual) @ Vt_manual
print(f"\nReconstruction error: {np.linalg.norm(A - A_reconstructed):.2e}")

In [ ]:
# LoRA: Low-Rank Adaptation - THE technique for efficient fine-tuning
# Instead of updating full weight matrix W (d x d), update ΔW = B @ A
# where B is (d x r) and A is (r x d), with r << d

d = 768  # typical hidden dimension (BERT-base)
r = 16   # LoRA rank (typical: 8, 16, 32)

# Original weight matrix
W = np.random.randn(d, d)

# LoRA matrices (much smaller!)
B = np.random.randn(d, r) * 0.01  # initialized small
A = np.random.randn(r, d) * 0.01

# Effective weight = W + ΔW = W + B @ A
delta_W = B @ A  # (d x r) @ (r x d) = (d x d)

print(f"Original parameters:  {d * d:,} ({d}x{d})")
print(f"LoRA parameters:      {d * r + r * d:,} ({d}x{r} + {r}x{d})")
print(f"Parameter reduction:  {d*d / (d*r + r*d):.1f}x fewer parameters")
print(f"Memory savings:       {(1 - (d*r + r*d)/(d*d)) * 100:.1f}%")

## 5. Softmax and Numerical Stability

The softmax function converts raw scores (logits) into a probability distribution.
It's used at the output of every classification network and inside every attention mechanism.

In [ ]:
def softmax_naive(z: np.ndarray) -> np.ndarray:
    """Naive softmax - WILL overflow for large values!"""
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

def softmax_stable(z: np.ndarray) -> np.ndarray:
    """Numerically stable softmax using log-sum-exp trick.
    
    Key insight: softmax(z) = softmax(z - c) for any constant c.
    Choosing c = max(z) prevents overflow.
    """
    z_shifted = z - np.max(z)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z)

# Safe case
z_safe = np.array([1.0, 2.0, 3.0])
print("Safe input:", z_safe)
print(f"  Naive:  {softmax_naive(z_safe)}")
print(f"  Stable: {softmax_stable(z_safe)}")

# Dangerous case: large logits (common in practice!)
z_dangerous = np.array([1000.0, 1001.0, 1002.0])
print(f"\nDangerous input: {z_dangerous}")
print(f"  Naive:  {softmax_naive(z_dangerous)}")  # NaN!
print(f"  Stable: {softmax_stable(z_dangerous)}")  # Works correctly

In [ ]:
# Cross-entropy loss: THE loss function for classification
def cross_entropy_loss(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute cross-entropy loss.
    
    H(p, q) = -Σ p(x) log q(x)
    
    For one-hot y_true, this simplifies to: -log(q(y_true))
    """
    # Clip predictions to avoid log(0)
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    return -np.sum(y_true * np.log(y_pred))

# Example: 3-class classification
y_true = np.array([0, 0, 1])  # ground truth: class 2

# Good prediction
y_pred_good = softmax_stable(np.array([-1.0, 0.0, 3.0]))
# Bad prediction
y_pred_bad = softmax_stable(np.array([3.0, 0.0, -1.0]))

print(f"True label: class 2")
print(f"Good prediction: {y_pred_good} → loss = {cross_entropy_loss(y_true, y_pred_good):.4f}")
print(f"Bad prediction:  {y_pred_bad} → loss = {cross_entropy_loss(y_true, y_pred_bad):.4f}")
print("\n→ Higher loss = worse prediction (model needs to update weights)")

## 6. Information Theory Implementations

Entropy, cross-entropy, and KL divergence are the foundation of all loss functions in ML.

In [ ]:
def entropy(p: np.ndarray) -> float:
    """Shannon entropy: H(p) = -Σ p(x) log p(x)"""
    p = p[p > 0]  # Remove zeros to avoid log(0)
    return -np.sum(p * np.log2(p))

def kl_divergence(p: np.ndarray, q: np.ndarray) -> float:
    """KL Divergence: D_KL(p || q) = Σ p(x) log(p(x)/q(x))
    
    Measures how different q is from p.
    NOT symmetric: D_KL(p||q) ≠ D_KL(q||p)
    """
    mask = p > 0
    return np.sum(p[mask] * np.log2(p[mask] / q[mask]))

def cross_entropy_info(p: np.ndarray, q: np.ndarray) -> float:
    """Cross-entropy: H(p, q) = -Σ p(x) log q(x) = H(p) + D_KL(p||q)"""
    mask = p > 0
    return -np.sum(p[mask] * np.log2(q[mask]))

# Demonstrate the relationship: H(p,q) = H(p) + D_KL(p||q)
p = np.array([0.7, 0.2, 0.1])  # true distribution
q = np.array([0.4, 0.4, 0.2])  # model distribution

H_p = entropy(p)
H_pq = cross_entropy_info(p, q)
D_KL = kl_divergence(p, q)

print(f"H(p)           = {H_p:.4f} bits")
print(f"D_KL(p || q)   = {D_KL:.4f} bits")
print(f"H(p) + D_KL    = {H_p + D_KL:.4f} bits")
print(f"H(p, q)        = {H_pq:.4f} bits")
print(f"\n✓ H(p,q) = H(p) + D_KL(p||q) verified!")

# Show asymmetry of KL divergence
print(f"\nD_KL(p || q) = {kl_divergence(p, q):.4f}")
print(f"D_KL(q || p) = {kl_divergence(q, p):.4f}")
print("→ KL divergence is NOT symmetric!")

## 7. Broadcasting and Batched Operations

Understanding broadcasting is critical for writing efficient PyTorch/NumPy code.
Most AI bugs come from shape mismatches.

In [ ]:
# Broadcasting rules:
# 1. If arrays have different #dims, prepend 1s to the smaller shape
# 2. Arrays with size 1 in a dimension act as if they match the other array's size
# 3. If sizes don't match and neither is 1 → error

# Example: adding bias to batch of outputs
batch_output = np.random.randn(32, 10)  # (batch_size=32, output_dim=10)
bias = np.random.randn(10)               # (output_dim=10,) → broadcasts to (1, 10)

result = batch_output + bias  # (32, 10) + (10,) → (32, 10)
print(f"Batch output: {batch_output.shape}")
print(f"Bias:         {bias.shape}")
print(f"Result:       {result.shape}")

# Common broadcasting patterns in AI:
print("\n--- Common Broadcasting Patterns ---")

# 1. Softmax temperature scaling
logits = np.random.randn(8, 1000)  # (batch, vocab_size)
temperature = 0.7                   # scalar
scaled = logits / temperature       # broadcasts scalar across all elements
print(f"Temperature scaling: {logits.shape} / scalar → {scaled.shape}")

# 2. Layer normalization (mean/var per sample)
x = np.random.randn(4, 768)        # (batch, hidden_dim)
mean = x.mean(axis=1, keepdims=True)  # (4, 1)
std = x.std(axis=1, keepdims=True)    # (4, 1)
x_normalized = (x - mean) / (std + 1e-5)  # (4, 768) - (4, 1) → broadcasts!
print(f"Layer norm: {x.shape} - {mean.shape} → {x_normalized.shape}")

## Summary

| Concept | AI Application |
|---------|---------------|
| Dot product | Neuron computation, attention scores |
| Matrix multiply | Every layer's forward pass |
| Norms | Regularization, gradient clipping |
| Cosine similarity | Embedding search, RAG retrieval |
| Eigendecomposition | PCA, spectral methods |
| SVD | LoRA, dimensionality reduction |
| Softmax | Classification output, attention weights |
| Cross-entropy | THE classification loss function |
| KL divergence | VAE loss, knowledge distillation, RLHF |

**Next**: [Optimization Algorithms](02_optimization_algorithms.ipynb)